# 📝 Guião 10
[Chat com modelo IA utilizado para realização deste guião](https://gemini.google.com/share/6d843f59c3d2)

### [Tarefa] Geração da matriz A

> A especificação do **FrodoKEM** recorre ao `SHAKE256` ou `AES` para expandir a matriz aleatória $A$ a partir de uma semente com 128 bit ($\seedA$). Para efeitos deste guião, vamos recorrer directamente às bibliotecas > do **SageMath** ou **Python** para esse efeito.
> 
> Deve então codificar a função `genMat`, que recebe a semente `seedA`, e retorna uma matriz aleatória de dimensão $n \times n$ com elementos amostrados com distribuição uniforme em $\mathbb{R}$.

In [28]:
def genMat(seedA, n, mod):
    set_random_seed(seedA)
    Rq = Integers(mod)
    return random_matrix(Rq, n, n)

### [Tarefa] Geração de segredos/ruído

> Para a geração dos segredos/ruído, deve-se fazer uso de uma distribuição que aproxime uma distribuição Gaussiana centrada na origem. Para tal é usual recorrer-se a tabelas pré-computadas com as probabilidades acumuladas dos valores em torno da origem. A tabela respectiva para a distribuição $\chi$ dos parâmetros acima apresentados é:
>
> ```python
> T_CHI640 = [4643, 13363, 20579, 25843, 29227, 31145, 32103, 32525, 32689, 32745, 32762, 32767]
> ```
>
> Esta tabela foi gerada para produzir uma aproximação da distribuição Gaussiana centrada na origem com desvio padrão $\sigma \approx 2.8$. Permite gerar um valor com a distribuição pretendida da seguinte forma:
> 
> - gera-se um número aleatório $x$ com a distribuição uniforme na gama $[0 .. 2^{k}[$, onde $k=15$ neste caso (bits de precisão adoptados)
> - determina-se a posição $i$ da tabela em que aparece o primeiro valor não inferior a $x$
> - amostra-se um bit adicional para o sinal ($s$)
> - o valor retornado é então $(1-2*s) * i$
> 
> Por inspecção da tabela se percebe que os valores mais prováveis são em torno do $0$. De resto, não é sequer possível sairem valores com valor absoluto superior a $11$.
> 
> Defina a função `chiSample`, que receba a tabela com as probabilidades acumuladas e as dimensões da matriz a produzir. Como resultado, retorna a matriz com elementos gerados de acordo com a distribuição descrita.

In [29]:
def chiSample(T_CHI, rows, cols, precision):
    res = matrix(ZZ, rows, cols)
    limit = 2**precision
    
    for i in range(rows):
        for j in range(cols):
            x = ZZ.random_element(0, limit)
            sample_value = 0
            for idx, val in enumerate(T_CHI):
                if val > x:
                    sample_value = idx
                    break
            sign = 1 if randint(0, 1) == 0 else -1
            res[i, j] = sign * sample_value
            
    return res


### [Tarefa] Codificação da mensagem

> A mensagem (128 bit) é codificada numa matriz $\bar{n}\times\bar{n}$, sendo que cada elemento dessa matriz codifica dois ($B$) bits. Esses bits são codificados escalando-os no anel $\mathbb{R}$, i.e., para $x \in\{0,1,2,3\}$, considera-se o elemento
>
> $$ x \cdot \frac{q}{2^B} $$
>
> A descodificação de um valor do anel numa mensagem é feita por arredondamento, i.e., para $x\in\mathbb{Z}_q$, devolve-se
>
> $$ \mathrm{round}\left( \frac{x\cdot 2^B}{q} \right) $$
>
> Defina as funções `encode` e `decode` que realizam a codificação/descodificação da mensagem como acima detalhado.

In [52]:
def encode(message_bits, n_bar, q, B):
    Rq = Integers(q)
    M = matrix(Rq, n_bar, n_bar)
    factor = q // (2**B)
    
    idx = 0
    for i in range(n_bar):
        for j in range(n_bar):
            chunk = message_bits[idx]
            M[i, j] = chunk * factor
            idx += 1
    return M

def decode(M, q, B):
    n_bar = M.nrows()
    res = []
    factor = q / (2**B)
    
    for i in range(n_bar):
        for j in range(n_bar):
            val = round(float(M[i, j].lift()) / float(factor))
            res.append(int(val) % (2**B))
    return res

### [Tarefa] PKE

> Estamos finalmente em condições de codificar os algoritmos de PKE:
> 
> - $\mathrm{KeyGen}() =$
>   - $\mathsf{seedA} \stackrel{\\$}{\leftarrow} \{0 .. q-1\}$
>   - $A = \mathrm{genMat}(\mathsf{seedA})$
>   - $S \stackrel{\\$}{\leftarrow} \mathrm{chiSample}(n, \bar{n})$
>   - $E \stackrel{\\$}{\leftarrow} \mathrm{chiSample}(n, \bar{n})$
>   - $B = A\cdot S + E$
>   - $\mathsf{pk} = \langle \mathsf{seedA}, B \rangle$
>   - $\mathsf{sk} = S$
>   - **return** $\langle \mathsf{sk}, \mathsf{pk}\rangle$
>
> - $\mathrm{Enc}(\mathrm{pk}, \mathrm{m}) =$
>   - $\langle \mathsf{seedA}, B\rangle = \mathsf{pk}$
>   - $A = \mathrm{genMat}(\mathsf{seedA})$
>   - $S' \stackrel{\\$}{\leftarrow} \mathrm{chiSample}(\bar{n}, n)$
>   - $E' \stackrel{\\$}{\leftarrow} \mathrm{chiSample}(\bar{n}, n)$
>   - $E'' \stackrel{\\$}{\leftarrow} \mathrm{chiSample}(\bar{n}, \bar{n})$
>   - $C_1 = S' \cdot A + E'$
>   - $V' = S' \cdot B + E''$
>   - $M = \mathrm{encode}(\mathrm{m})$
>   - $C_2 = V' + M$
>   - **return** $\langle C_1, C_2\rangle$
> 
> - $\mathrm{Dec}(\mathsf{sk}, C) =$
>   - $\langle C_1, C_2\rangle = C$
>   - $S = \mathrm{sk}$
>   - $A = \mathrm{genMat}(\mathrm{seedA})$
>   - $V = C_1 \cdot S$
>   - $M = C_2 - V$
>   - $\mathrm{m}' = \mathrm{decode}(M)$
>   - **Return** $\mathrm{m}'$

In [47]:
class PKE:
    def __init__(self):
        self.n = 640
        self.q = 2**15
        self.n_bar = 8
        self.B = 2
        self.table = [4643, 13363, 20579, 25843, 29227, 31145, 32103, 32525, 32689, 32745, 32762, 32767]
        self.precision = 15
        self.Rq = Integers(self.q)

    def keygen(self):
        seedA = randint(0, self.q - 1)
        A = genMat(seedA, self.n, self.q)
        S = chiSample(self.table, self.n, self.n_bar, self.precision).change_ring(self.Rq)
        E = chiSample(self.table, self.n, self.n_bar, self.precision).change_ring(self.Rq)
        B_mat = A * S + E
        return (seedA, B_mat), S

    def encrypt(self, pk, message_chunks):
        seedA, B_mat = pk
        A = genMat(seedA, self.n, self.q)
        Sp = chiSample(self.table, self.n_bar, self.n, self.precision).change_ring(self.Rq)
        Ep = chiSample(self.table, self.n_bar, self.n, self.precision).change_ring(self.Rq)
        Epp = chiSample(self.table, self.n_bar, self.n_bar, self.precision).change_ring(self.Rq)
        
        C1 = Sp * A + Ep
        Vp = Sp * B_mat + Epp
        M = encode(message_chunks, self.n_bar, self.q, self.B)
        C2 = Vp + M
        return (C1, C2)

    def decrypt(self, sk, ciphertext):
        C1, C2 = ciphertext
        S = sk
        V = C1 * S
        M_approx = C2 - V
        return decode(M_approx, self.q, self.B)

### [Tarefa] miniPKE
> Adapte a implementação anterior para os novos parâmetros.

In [46]:
class miniPKE:
    def __init__(self):
        self.n = 64
        self.q = 2**10
        self.n_bar = 8
        self.B = 2
        self.table = [6536, 14465, 16234, 16379, 16384]
        self.precision = 14
        self.Rq = Integers(self.q)

    def keygen(self):
        seedA = randint(0, self.q - 1)
        A = genMat(seedA, self.n, self.q)
        S = chiSample(self.table, self.n, self.n_bar, self.precision).change_ring(self.Rq)
        E = chiSample(self.table, self.n, self.n_bar, self.precision).change_ring(self.Rq)
        B_mat = A * S + E
        return (seedA, B_mat), S

    def encrypt(self, pk, message_chunks):
        seedA, B_mat = pk
        A = genMat(seedA, self.n, self.q)
        Sp = chiSample(self.table, self.n_bar, self.n, self.precision).change_ring(self.Rq)
        Ep = chiSample(self.table, self.n_bar, self.n, self.precision).change_ring(self.Rq)
        Epp = chiSample(self.table, self.n_bar, self.n_bar, self.precision).change_ring(self.Rq)
        
        C1 = Sp * A + Ep
        Vp = Sp * B_mat + Epp
        M = encode(message_chunks, self.n_bar, self.q, self.B)
        C2 = Vp + M
        return (C1, C2)

    def decrypt(self, sk, ciphertext):
        C1, C2 = ciphertext
        S = sk
        V = C1 * S
        M_approx = C2 - V
        return decode(M_approx, self.q, self.B)

### [Tarefa] Impacto do ruído

> Ajuste a implementação do **miniPKE** para usar a distribuição de ruído do esquema original (`T_CHI640`). Verifique e comente os impactos na correcção do algoritmo.

In [61]:
class miniPKE640:
    def __init__(self):
        self.n = 64
        self.q = 2**10
        self.n_bar = 8
        self.B = 2
        self.table = [4643, 13363, 20579, 25843, 29227, 31145, 32103, 32525, 32689, 32745, 32762, 32767]
        self.precision = 15
        self.Rq = Integers(self.q)

    def keygen(self):
        seedA = randint(0, self.q - 1)
        A = genMat(seedA, self.n, self.q)
        S = chiSample(self.table, self.n, self.n_bar, self.precision).change_ring(self.Rq)
        E = chiSample(self.table, self.n, self.n_bar, self.precision).change_ring(self.Rq)
        B_mat = A * S + E
        return (seedA, B_mat), S

    def encrypt(self, pk, message_chunks):
        seedA, B_mat = pk
        A = genMat(seedA, self.n, self.q)
        Sp = chiSample(self.table, self.n_bar, self.n, self.precision).change_ring(self.Rq)
        Ep = chiSample(self.table, self.n_bar, self.n, self.precision).change_ring(self.Rq)
        Epp = chiSample(self.table, self.n_bar, self.n_bar, self.precision).change_ring(self.Rq)
        
        C1 = Sp * A + Ep
        Vp = Sp * B_mat + Epp
        M = encode(message_chunks, self.n_bar, self.q, self.B)
        C2 = Vp + M
        return (C1, C2)

    def decrypt(self, sk, ciphertext):
        C1, C2 = ciphertext
        S = sk
        V = C1 * S
        M_approx = C2 - V
        return decode(M_approx, self.q, self.B)

### 📊 Resultados

In [62]:
def testar_esquema(kem_instance, nome_esquema, num_testes=10):
    print(f"{nome_esquema}")
    print(f"Parâmetros: n={kem_instance.n}, q={kem_instance.q}, B={kem_instance.B}")
    
    # --- 1. Teste de Codificação Base (Sanity Check) ---
    print("\n[1] A testar Encode / Decode (Matriz Completa)...")
    sucesso_codificacao = True
    try:
        # Gerar uma mensagem de teste aleatória (n_bar^2 valores entre 0 e 2^B - 1)
        limite_msg = (2**kem_instance.B) - 1
        msg_teste = [randint(0, limite_msg) for _ in range(kem_instance.n_bar**2)]
        
        encoded = encode(msg_teste, kem_instance.n_bar, kem_instance.q, kem_instance.B)
        decoded = decode(encoded, kem_instance.q, kem_instance.B)
        
        if msg_teste != decoded:
            print(f"  -> ERRO: A descodificação não bate com a original.")
            sucesso_codificacao = False
    except Exception as e:
        print(f"  -> ERRO na execução de encode/decode: {e}")
        sucesso_codificacao = False
            
    if sucesso_codificacao:
        print("  -> OK! Codificação e descodificação base a funcionar.")

    # --- 2. Teste do Algoritmo PKE ---
    print(f"\n[2] A testar ciclo PKE Completo ({num_testes} iterações)...")
    sucessos_pke = 0
    
    for i in range(num_testes):
        try:
            # 2.1 Gerar Chaves
            pk, sk = kem_instance.keygen()
            
            # 2.2 Gerar mensagem
            msg_original = [randint(0, (2**kem_instance.B) - 1) for _ in range(kem_instance.n_bar**2)]
            
            # 2.3 Cifrar e Decifrar
            ciphertext = kem_instance.encrypt(pk, msg_original)
            msg_decifrada = kem_instance.decrypt(sk, ciphertext)
            
            # 2.4 Verificar integridade
            if msg_original == msg_decifrada:
                sucessos_pke += 1
                
        except Exception as e:
            print(f"  -> ERRO fatal na execução do PKE (Iteração {i+1}): {e}")
            break

    taxa_sucesso = float(sucessos_pke / num_testes) * 100.0

    print(f"\nResultados")
    print(f"Sucessos: {sucessos_pke}/{num_testes} ({taxa_sucesso:.1f}%)")
    print(f"_"*40)

frodo_oficial = PKE()
testar_esquema(frodo_oficial, "\nFrodoKEM-640", num_testes=10)

minifrodo = miniPKE()
testar_esquema(minifrodo, "\nminiFrodoKem", num_testes=10)

minifrodo = miniPKE640()
testar_esquema(minifrodo, "\nminiFrodoKem com T_CHI640", num_testes=1000)


FrodoKEM-640
Parâmetros: n=640, q=32768, B=2

[1] A testar Encode / Decode (Matriz Completa)...
  -> OK! Codificação e descodificação base a funcionar.

[2] A testar ciclo PKE Completo (10 iterações)...

Resultados
Sucessos: 10/10 (100.0%)
________________________________________

miniFrodoKem
Parâmetros: n=64, q=1024, B=2

[1] A testar Encode / Decode (Matriz Completa)...
  -> OK! Codificação e descodificação base a funcionar.

[2] A testar ciclo PKE Completo (10 iterações)...

Resultados
Sucessos: 10/10 (100.0%)
________________________________________

miniFrodoKem com T_CHI640
Parâmetros: n=64, q=1024, B=2

[1] A testar Encode / Decode (Matriz Completa)...
  -> OK! Codificação e descodificação base a funcionar.

[2] A testar ciclo PKE Completo (1000 iterações)...

Resultados
Sucessos: 0/1000 (0.0%)
________________________________________


Ao ser aumentado o ruído, aumenta drasticamente a taxa de insucesso, tal como prova a tabela do guião. O **miniFrodoKem** com $q=2^{10}$ não tem "espaço" matemático (módulo suficientemente grande) para acomodar o ruído mais agressivo do **FrodoKEM-640** sem destruir a mensagem no processo.

Mas brincado um pouco com o `q` é possível verificar que, se aumentarmos o expoente de 10 para 11, ou seja, $q=2^{11}$ já teriamos uma taxa de 50% de sucesso.